# DCGAN sur Fashion-MNIST


In [ ]:
%pip install -q torch torchvision lightning matplotlib pandas

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import lightning.pytorch as pl

from torch.utils.data import DataLoader, TensorDataset
from torchvision.utils import make_grid, save_image

# ====================== CONFIGURATION ======================
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ====================== CHEMINS ======================
DATA_DIR = Path('/kaggle/input/datasets/mlonjoansafagnibo/generative')
OUTPUT_DIR = Path('/kaggle/working/outputs/dcgan_fashion_mnist')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ====================== PARAMETRES ======================
LATENT_DIM = 100
IMAGE_SIZE = 28
CHANNELS = 1
BATCH_SIZE = 128
LR = 1e-3
BETA1 = 0.5
BETA2 = 0.999
MAX_EPOCHS = 45

print('CUDA available:', torch.cuda.is_available())
print('Output dir:', OUTPUT_DIR)

In [ ]:
TRAIN_CSV = DATA_DIR / 'fashion-mnist_train.csv'

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f'Missing Fashion-MNIST CSV: {TRAIN_CSV}')

train_df = pd.read_csv(TRAIN_CSV)
labels = torch.tensor(train_df.iloc[:, 0].values, dtype=torch.long)
images = torch.tensor(train_df.iloc[:, 1:].values, dtype=torch.float32)
images = images.view(-1, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
images = (images / 127.5) - 1.0

train_dataset = TensorDataset(images, labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print('Train images:', len(train_dataset))
print('Image tensor shape:', images.shape)

In [ ]:
real_images, labels = next(iter(train_loader))
grid = make_grid((real_images[:64] + 1) / 2, nrow=8)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
plt.axis('off')
plt.title('Fashion-MNIST - vraies images')
plt.show()

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, IMAGE_SIZE * IMAGE_SIZE),
            nn.Tanh(),
        )

    def forward(self, z):
        images = self.model(z)
        return images.view(z.size(0), CHANNELS, IMAGE_SIZE, IMAGE_SIZE)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMAGE_SIZE * IMAGE_SIZE, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, images):
        return self.model(images)

In [ ]:
class FashionDCGAN(pl.LightningModule):
    def __init__(self, latent_dim=100, lr=2e-4, beta1=0.5, beta2=0.999):
        super().__init__()
        self.save_hyperparameters()
        self.generator = Generator(latent_dim)
        self.discriminator = Discriminator()
        self.criterion = nn.BCEWithLogitsLoss()
        self.automatic_optimization = False
        self.register_buffer('fixed_z', torch.randn(64, latent_dim))

    def forward(self, z):
        return self.generator(z)

    def training_step(self, batch, batch_idx):
        real_images, _ = batch
        batch_size = real_images.size(0)
        opt_g, opt_d = self.optimizers()

        real_labels = torch.ones(batch_size, 1, device=self.device)
        fake_labels = torch.zeros(batch_size, 1, device=self.device)

        z = torch.randn(batch_size, self.hparams.latent_dim, device=self.device)
        fake_images = self.generator(z).detach()
        real_logits = self.discriminator(real_images)
        fake_logits = self.discriminator(fake_images)
        d_loss = self.criterion(real_logits, real_labels) + self.criterion(fake_logits, fake_labels)

        opt_d.zero_grad()
        self.manual_backward(d_loss)
        opt_d.step()

        z = torch.randn(batch_size, self.hparams.latent_dim, device=self.device)
        generated_images = self.generator(z)
        generated_logits = self.discriminator(generated_images)
        g_loss = self.criterion(generated_logits, real_labels)

        opt_g.zero_grad()
        self.manual_backward(g_loss)
        opt_g.step()

        self.log('g_loss', g_loss, prog_bar=True)
        self.log('d_loss', d_loss, prog_bar=True)

    def configure_optimizers(self):
        opt_g = torch.optim.Adam(
            self.generator.parameters(),
            lr=self.hparams.lr,
            betas=(self.hparams.beta1, self.hparams.beta2),
        )
        opt_d = torch.optim.Adam(
            self.discriminator.parameters(),
            lr=self.hparams.lr,
            betas=(self.hparams.beta1, self.hparams.beta2),
        )
        return [opt_g, opt_d]

    def sample_images(self):
        self.generator.eval()
        with torch.no_grad():
            images = self.generator(self.fixed_z.to(self.device))
        self.generator.train()
        return images

In [ ]:
class ImageSampler(pl.Callback):
    def __init__(self, output_dir, every_n_epochs=1):
        super().__init__()
        self.output_dir = Path(output_dir)
        self.every_n_epochs = every_n_epochs
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch + 1
        if epoch % self.every_n_epochs != 0:
            return
        images = (pl_module.sample_images() + 1) / 2
        save_image(images, self.output_dir / f'epoch_{epoch:03d}.png', nrow=8)

In [ ]:
gan = FashionDCGAN(
    latent_dim=LATENT_DIM,
    lr=LR,
    beta1=BETA1,
    beta2=BETA2,
)

precision = '16-mixed' if torch.cuda.is_available() else '32-true'

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='auto',
    devices=1,
    precision=precision,
    callbacks=[ImageSampler(OUTPUT_DIR, every_n_epochs=1)],
    logger=False,
)

trainer.fit(gan, train_loader)

In [ ]:
gan.eval()

with torch.no_grad():
    z = torch.randn(64, LATENT_DIM, device=gan.device)
    generated_images = gan.generator(z).cpu()

generated_images = (generated_images + 1) / 2
grid = make_grid(generated_images, nrow=8)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
plt.axis('off')
plt.title('Images generees par le GAN')
plt.show()

In [ ]:
print('Images sauvegardees dans :', OUTPUT_DIR)